In [44]:
import google.auth

credentials, project = google.auth.default()
print(credentials, project)

<google.oauth2.service_account.Credentials object at 0xffff088cacc0> wagon-ds-2026


In [45]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from google.cloud import storage
import io

In [46]:
from google.auth import impersonated_credentials

service_account = "le-wagon-bootcamp@le-wagon-2303.iam.gserviceaccount.com"
signing_credentials = impersonated_credentials.Credentials(
    source_credentials=credentials,
    target_principal=service_account,
    target_scopes=["https://www.googleapis.com/auth/cloud-platform"],
    lifetime=3600,
)


bucket_client = storage.Client(project='le-wagon-2303', credentials=signing_credentials)
bucket = bucket_client.bucket('sm-optimizer-processed')


In [47]:
# get all bucket video keys
yt_video_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="videos/youtube")
yt_video_keys = [f"{blob.name.split('/')[-1].removesuffix('.mp4')}" for blob in yt_video_blobs]

tt_video_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="videos/tiktok")
tt_video_keys = [f"{blob.name.split('/')[-1].removesuffix('.mp4')}" for blob in tt_video_blobs]

fb_video_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="videos/facebook")
fb_video_keys = [f"{blob.name.split('/')[-1].removesuffix('.mp4')}" for blob in fb_video_blobs]

ig_video_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="videos/instagram")
ig_video_keys = [f"{blob.name.split('/')[-1].removesuffix('.mp4')}" for blob in ig_video_blobs]


all_video_keys = yt_video_keys + tt_video_keys + fb_video_keys + ig_video_keys
print(all_video_keys[0], all_video_keys[-1])
print(len(all_video_keys))

--CmvoDexw0 DWaJAz8D07F
12718


In [48]:
### existing description.json (currently empty)
yt_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/youtube")
yt_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in yt_description_blobs]

tt_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/tiktok")
tt_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in tt_description_blobs]

fb_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/facebook")
fb_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in fb_description_blobs]

ig_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/instagram")
ig_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in ig_description_blobs]

all_description_keys = yt_description_keys + tt_description_keys + fb_description_keys + ig_description_keys
print(all_description_keys[0], all_description_keys[-1])
print('descriptions in bucket:', len(all_description_keys))

 
descriptions in bucket: 64


In [49]:
def undescribed_keys(description_keys, video_keys):
    """
    returns keys that are not found in 'description_keys' but exist in 'video_keys'
    """
    return set(video_keys) - set(description_keys)

missing_yt = undescribed_keys(yt_description_keys, yt_video_keys)
print(len(missing_yt))
missing_tt = undescribed_keys(tt_description_keys, tt_video_keys)
print(len(missing_tt))
missing_fb = undescribed_keys(fb_description_keys, fb_video_keys)
print(len(missing_fb))
missing_ig = undescribed_keys(ig_description_keys, ig_video_keys)
print(len(missing_ig))

print('sum', len(missing_yt) + len(missing_tt) + len(missing_fb) + len(missing_ig))
print('sample', list(missing_yt)[0])

1474
1995
4468
4721
sum 12658
sample E7In0LkEf_o


In [50]:
def undescribed_blobs(undescribed_keys, platform):
    """
    Given an iterable of video keys missing descriptions and a platform name
    ('youtube', 'tiktok', 'facebook', 'instagram'), return the matching Blob
    objects from the 'sm-optimizer-processed' bucket.
    """
    blobs = bucket_client.list_blobs(
        'sm-optimizer-processed',
        prefix=f"videos/{platform}",
    )
    return [
        blob for blob in blobs
        if blob.name.split('/')[-1].removesuffix('.mp4') in undescribed_keys
    ]

yt_blobs  = undescribed_blobs(missing_yt, 'youtube')
print(len(yt_blobs))
tt_blobs  = undescribed_blobs(missing_tt, 'tiktok')
print(len(tt_blobs))
fb_blobs  = undescribed_blobs(missing_fb, 'facebook')
print(len(fb_blobs))
ig_blobs  = undescribed_blobs(missing_ig, 'instagram')
print(len(ig_blobs))

print('sum ', len(yt_blobs) + len(tt_blobs) + len(fb_blobs) + len(ig_blobs))
sample = yt_blobs[0]
print('sample ', sample)

1474
1995
4468
4721
sum  12658
sample  <Blob: sm-optimizer-processed, videos/youtube/-veq8XsBreY.mp4, 1787816971459824>


In [56]:
MAX_VIDEO_BYTES = 15 * 1024 * 1024  # 15 MiB — Vertex signed-URL fetch cap (`max_bytes_fetched: 15728640`)
VIDEO_PREFIXES = ("videos/youtube", "videos/tiktok", "videos/facebook", "videos/instagram")

oversized = []
total_blobs = 0

for prefix in VIDEO_PREFIXES:
    for blob in bucket_client.list_blobs("sm-optimizer-processed", prefix=prefix):
        total_blobs += 1
        if blob.size is not None and blob.size > MAX_VIDEO_BYTES:
            key = blob.name.split("/")[-1].removesuffix(".mp4")
            oversized.append((blob.size, key, blob.name))

oversized.sort(reverse=True)

print(
    f"oversized: {len(oversized)} of {total_blobs} video blobs exceed "
    f"{MAX_VIDEO_BYTES // (1024 * 1024)} MB ({MAX_VIDEO_BYTES} bytes)"
)
print(f"keys ({len(oversized)}):")
for size, key, name in oversized:
    print(f"  {size / (1024 * 1024):>8.1f} MB  {key}")


oversized: 117 of 12718 video blobs exceed 15 MB (15728640 bytes)
keys (117):
     153.0 MB  269531545700896
     143.0 MB  817480308020096
     109.3 MB  1510423103389574
      81.2 MB  1677581550025256
      62.9 MB  7617291565530205461
      61.7 MB  775847037834534
      59.2 MB  DV2-p01irNI
      43.7 MB  1725307861595031
      43.0 MB  1210099139983070
      41.7 MB  295806973402511
      41.3 MB  482276981504686
      38.7 MB  CsikVAdq5ft
      36.1 MB  634447514760411
      36.1 MB  880779047651168
      35.2 MB  7589836112911191313
      34.9 MB  DS6QD0GFaxm
      34.1 MB  DCaIJB3Tpex
      33.5 MB  DDJThlYKMZS
      33.5 MB  1795389677666464
      32.8 MB  592907099843719
      32.4 MB  661820630084873
      32.2 MB  DVwYS6lEuHu
      29.0 MB  DPQbPxCET1D
      28.6 MB  DCRQWGQyNtO
      28.3 MB  4222174734665701
      27.4 MB  1706236626467719
      27.1 MB  369773072648638
      26.8 MB  2208805009645302
      26.3 MB  C2ivqi6vrSV
      25.5 MB  622943836704950
      25.3 M

In [52]:
from datetime import timedelta
from google import genai
from google.genai import types
import json

MODEL = "gemini-3.6-flash"
PROMPT = """
You are a video analysis and content classification system. 
Do not include markdown, explanations, commentary, or any text outside the JSON object.
Return JSON only.

Your output has two purposes:
1. Produce a play-by-play(dont include timestamps) / transcript of the video.
2. Classify the video using the controlled vocabularies below.

GENERAL CLASSIFICATION RULES:
- Only assign values that are clearly supported by the video.
- Use ONLY the allowed values listed below UNLESS specified in the features section exclusively.
- Do not invent values, synonyms, or related terms.
- A feature may contain multiple values when multiple values clearly apply.
- If no allowed value applies, return ["undefined"].
- Every classification feature must always be present.
- Preserve the exact spelling and casing of the allowed values.
- Do not infer identities solely from appearance unless the identity is explicitly established by the video, visible text, audio, or other clear evidence.

PLAY-BY-PLAY / TRANSCRIPT:

Return a chronological description of what happens in the video. 

If there is spoken dialogue:
- Transcribe the meaningful spoken content as accurately as possible.
- Preserve the order of speakers when they can be distinguished.
- Do not invent dialogue that is unclear or inaudible.

If there is little or no speech:
- Describe the sequence of important visible events instead.
- Include actions, people entering/leaving, major plays, reactions, celebrations,
  locations, and meaningful visual changes.
- Focus on what actually happens rather than giving a generic summary.

The play-by-play should be concise but sufficiently detailed to capture the important events, and the scene settings. If players appear on screen, the description should specify whether they’re male or female, whenever possible.

Do not speculate about events that cannot be observed.

FEATURES:

1. content_theme
Allowed values:
["haka", "training", "try", "tackle", "player story", "celebration", “rugby_skills”,”kick”,
"rivalry", "challenges", "run", "conversion", “non rugby related”]

2. format_access
Allowed values:
["highlight", "interview", "montage", "behind-the-scenes",
"announcement", "candid clip", "archive", "reaction",
"promotion", "ticket sales"]

3. people (interpolate as you see fit, in lower case)
Allowed values:
["{rugby player name}", "{country} rugby team", "{personality}", "{other sports mentioned}"]

4. brands (interpolate as you see fit, in lower case)
Allowed values:
["{sponsor brand/logo name seen/heard}"]

5. event (interpolate as you see fit, in lower case)
Allowed values:
["abxv", "aupiki", "bunnings npc", "classic ab",
"farah palmer cup", "heartland", "hsbc svns", "maori ab", "reilly",
"pac4", "provincial pride", "rugby league", "world cup",
"super rugby pacific", "u20", "trc", "u85", "olympics",
"all blacks", "sevens", "{any other tournament from rugby}"]

6. tone
Allowed values:
["excitement", "pride", "tension", "nostalgia", "humour",
"wholesome", "solemn", "sadness", "lighthearted", “provocative”]

7. context
Allowed values:
["gym", "changing room", "press conference", "announcement", "tour",
"pre-match", "match day", "post-match", "travel", "off-season",
"squad naming", "jersey reveal", “award”]

8. Overall team
[“women”, “men”, “veterans”, “maori”, “youth”]

9. Audio format
[“voice”, “song”, “none”, “ambient”, “other”]

OUTPUT FORMAT:

{
  "play_by_play": "",
  "content_theme": [],
  "format_access": [],
  "people": [],
  "brands": [],
  "event": [],
  "tone": [],
  "context": [],
  "overall_team": [],
  "audio_format": [],
}

Return JSON only.
"""

gen_client = genai.Client(
    vertexai=True,
    project="le-wagon-2303",
    credentials=signing_credentials,
    location="global",
)


def get_signed_url(blob):
    return bucket.blob(blob.name).generate_signed_url(
        version="v4",
        expiration=timedelta(minutes=15),
        method="GET",
    )

def describe_video(blob, platform):
    """
    Sign a v4 GET URL for `blob`, send it to Gemini with PROMPT, parse the
    JSON response, and upload it to:
        gs://sm-optimizer-processed/gemini-descriptions/{platform}/{key}.json
    Returns the uploaded blob path.
    """
    
    bucket_keys = {"facebook", "instagram", "youtube", "tiktok"}
    if platform not in bucket_keys:
        raise ValueError(f"platform must be one of {sorted(bucket_keys)}, got {platform!r}")
        
    signed_url = get_signed_url(blob)
    response = gen_client.models.generate_content(
        model=MODEL,
        contents=[
            types.Part.from_uri(
                file_uri=signed_url,
                mime_type="video/mp4",
            ),
            PROMPT,
        ],
    )

    text = response.text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    parsed = json.loads(text)

    key = blob.name.split("/")[-1].removesuffix(".mp4")
    blob_path = f"gemini-descriptions/{platform}/{key}.json"
    bucket.blob(blob_path).upload_from_string(
        json.dumps(parsed), content_type="application/json"
    )

    print(f"uploaded: {blob_path}")
    return blob_path

In [53]:
# describe_video(sample, 'youtube')

In [54]:
import asyncio
import itertools


async def _describe_one(blob, semaphore, platform):
    async with semaphore:
        return await asyncio.to_thread(describe_video, blob, platform)


async def process_batch(
    blobs,
    platform,
    max_concurrent=10,
    batch_size=10,
    total=100,
):
    """
    Async-batch driver over `describe_video`.

    - At most `max_concurrent` describe_video calls run at once (semaphore).
    - Iterates in chunks of `batch_size`; awaits the full chunk before starting the next.
    - Stops once `total` blobs have been processed.
    - Propagates exceptions (no return_exceptions) — one failure aborts the run.
    - Auto-skips blobs larger than 15 MiB (Vertex signed-URL cap) before scheduling.
    """
    MAX_VIDEO_BYTES = 15 * 1024 * 1024  # 15 MiB — Vertex signed-URL fetch cap
    before_count = len(blobs)
    blobs = [b for b in blobs if (b.size or 0) <= MAX_VIDEO_BYTES]
    skipped = before_count - len(blobs)
    if skipped:
        print(f"skipped {skipped} oversized blobs (>15 MiB) in {platform}")

    semaphore = asyncio.Semaphore(max_concurrent)
    blob_iter = iter(blobs)
    processed = 0

    while processed < total:
        chunk = list(itertools.islice(blob_iter, batch_size))
        if not chunk:
            break

        remaining = total - processed
        if len(chunk) > remaining:
            chunk = chunk[:remaining]

        tasks = [
            asyncio.create_task(_describe_one(blob, semaphore, platform))
            for blob in chunk
        ]
        results = await asyncio.gather(*tasks)
        processed += len(chunk)
        print(f"batch done — processed {processed}/{total}")

    return processed


In [55]:
processed = await process_batch(yt_blobs, "youtube")
print("done:", processed)


skipped 1 oversized blobs (>15 MiB) in youtube
uploaded: gemini-descriptions/youtube/1lzUQ7Rt9P0.json
uploaded: gemini-descriptions/youtube/1imxQBt4mpM.json
uploaded: gemini-descriptions/youtube/1PUxHcUhVNE.json
uploaded: gemini-descriptions/youtube/1GsiyKujZBY.json
uploaded: gemini-descriptions/youtube/1PE_Mt0jc8Y.json
uploaded: gemini-descriptions/youtube/1L6iIEiQywg.json
uploaded: gemini-descriptions/youtube/1OgiUa98JzU.json
uploaded: gemini-descriptions/youtube/1Z64Z7fo1n4.json
uploaded: gemini-descriptions/youtube/1m-_dIL1Psg.json
uploaded: gemini-descriptions/youtube/1ieaqXl749A.json
batch done — processed 10/100
uploaded: gemini-descriptions/youtube/1r4Mg7NQ1dI.json
uploaded: gemini-descriptions/youtube/2Gnjt7eyUwM.json
uploaded: gemini-descriptions/youtube/1p-8cunZogk.json
uploaded: gemini-descriptions/youtube/1uR54QU1iAo.json
uploaded: gemini-descriptions/youtube/1wOO-OD8zPA.json
uploaded: gemini-descriptions/youtube/1xjfbl6Mzyc.json
uploaded: gemini-descriptions/youtube/2BPiz